# ESM-2 Embeddings — Optimized Production Pipeline

**Notebook:** `notebooks/ESM2_Embeddings_Optimized.ipynb`  
**Model:** ESM-2 650M (`esm2_t33_650M_UR50D`)  
**Dataset:** 2,390 protein sequences (780 Lassa + 1,610 Ebola)  

### Features
- ✅ Automatic GPU / CPU detection
- ✅ Mixed-precision (FP16) inference on GPU
- ✅ Dynamic batch-size adjustment (OOM recovery)
- ✅ Incremental checkpointing — resume after interruption
- ✅ Progress bars with ETA
- ✅ Embedding statistics, clustering metrics, PCA visualisation
- ✅ All outputs saved to `data/embeddings/` and `results/figures/`


## Section 0 — Setup & Configuration

In [ ]:
# ── Install / upgrade required packages ──────────────────────────────
import subprocess, sys

PACKAGES = [
    'fair-esm',
    'biopython',
    'tqdm',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'scipy',
]

for pkg in PACKAGES:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
        check=False
    )

print('✓ Package installation complete')


In [ ]:
# ── Core imports ─────────────────────────────────────────────────────
import os
import json
import logging
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from Bio import SeqIO
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from scipy.spatial.distance import euclidean, cosine

warnings.filterwarnings('ignore')

# ── Logging ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger(__name__)

# ── Reproducibility ──────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Output directories ────────────────────────────────────────────────
DIRS = [
    'data/embeddings',
    'data/processed',
    'results/figures',
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

# ── Seaborn style ────────────────────────────────────────────────────
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 100})

# ── Device detection ─────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 70)
print('ESM-2 EMBEDDINGS — OPTIMIZED PIPELINE')
print('=' * 70)
print(f'\n✓ Device : {device}')
if device.type == 'cuda':
    print(f'✓ GPU    : {torch.cuda.get_device_name(0)}')
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ VRAM   : {total_mem:.1f} GB')
print(f'✓ Seed   : {SEED}')


## Section 1 — Load Sequences

In [ ]:
import urllib.request

REPO_OWNER  = 'Damilola-max'
REPO_NAME   = 'Comparative_Lassa_Ebola-Model'
BRANCH      = 'main'
BASE_URL    = f'https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/{BRANCH}'

FASTA_FILES = {
    'lassa': {
        'url':   f'{BASE_URL}/data/cleaned/lassa_cleaned.fasta',
        'local': 'data/processed/lassa_cleaned.fasta',
    },
    'ebola': {
        'url':   f'{BASE_URL}/data/cleaned/ebola_cleaned.fasta',
        'local': 'data/processed/ebola_cleaned.fasta',
    },
}

for virus, info in FASTA_FILES.items():
    local = Path(info['local'])
    if local.exists():
        print(f'✓ {virus.capitalize()} FASTA already present: {local}')
    else:
        try:
            urllib.request.urlretrieve(info['url'], local)
            print(f'✓ Downloaded {virus.capitalize()} FASTA → {local}')
        except Exception as exc:
            log.warning(f'Could not download {virus} FASTA: {exc}')
            print(f'⚠ Could not download {virus} FASTA — please copy it manually to {local}')


In [ ]:
def load_fasta(filepath: str, virus_label: str) -> List[Dict]:
    """
    Parse a FASTA file and return a list of sequence records.

    Parameters
    ----------
    filepath    : Path to the FASTA file.
    virus_label : Label to attach to every record ('Lassa' or 'Ebola').

    Returns
    -------
    List of dicts with keys: id, sequence, length, virus.
    """
    records: List[Dict] = []
    path = Path(filepath)
    if not path.exists():
        log.error(f'File not found: {filepath}')
        return records
    try:
        for rec in SeqIO.parse(str(path), 'fasta'):
            seq = str(rec.seq).upper()
            if not seq:                          # skip empty sequences
                continue
            records.append({
                'id':       rec.id,
                'sequence': seq,
                'length':   len(seq),
                'virus':    virus_label,
            })
        log.info(f'Loaded {len(records):,} sequences from {filepath}')
    except Exception as exc:
        log.error(f'Error parsing {filepath}: {exc}')
    return records

lassa_seqs = load_fasta(FASTA_FILES['lassa']['local'], 'Lassa')
ebola_seqs = load_fasta(FASTA_FILES['ebola']['local'], 'Ebola')
all_seqs   = lassa_seqs + ebola_seqs

print(f'\n✓ Lassa  : {len(lassa_seqs):,} sequences')
print(f'✓ Ebola  : {len(ebola_seqs):,} sequences')
print(f'✓ Total  : {len(all_seqs):,} sequences')

if not all_seqs:
    raise RuntimeError(
        'No sequences loaded. '
        'Please ensure the FASTA files exist in data/processed/.'
    )


## Section 2 — Sequence Length Analysis

In [ ]:
lassa_lens = [s['length'] for s in lassa_seqs]
ebola_lens = [s['length'] for s in ebola_seqs]

print('Sequence Length Summary')
print('-' * 50)
print(f'Lassa : mean={np.mean(lassa_lens):.0f}  std={np.std(lassa_lens):.0f}'
      f'  min={min(lassa_lens)}  max={max(lassa_lens)}')
print(f'Ebola : mean={np.mean(ebola_lens):.0f}  std={np.std(ebola_lens):.0f}'
      f'  min={min(ebola_lens)}  max={max(ebola_lens)}')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Sequence Length Distribution', fontsize=14, fontweight='bold')

# Histogram
ax = axes[0]
ax.hist(lassa_lens, bins=40, alpha=0.6, color='#FF6B6B', label='Lassa')
ax.hist(ebola_lens, bins=40, alpha=0.6, color='#4ECDC4', label='Ebola')
ax.set_xlabel('Length (amino acids)', fontweight='bold')
ax.set_ylabel('Count', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Violin plot
ax = axes[1]
df_len = pd.DataFrame({
    'Length': lassa_lens + ebola_lens,
    'Virus':  ['Lassa'] * len(lassa_lens) + ['Ebola'] * len(ebola_lens),
})
sns.violinplot(
    data=df_len, x='Virus', y='Length', ax=ax,
    palette={'Lassa': '#FF6B6B', 'Ebola': '#4ECDC4'},
)
ax.set_ylabel('Length (amino acids)', fontweight='bold')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
fig_path = 'results/figures/01_sequence_length.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'\n✓ Saved: {fig_path}')
plt.show()


## Section 3 — Load ESM-2 Model

In [ ]:
try:
    import esm as fair_esm
except ImportError:
    raise ImportError(
        'The `fair-esm` package is not installed. '
        'Run: pip install fair-esm'
    )

MODEL_NAME   = 'esm2_t33_650M_UR50D'
REPR_LAYER   = 33          # last transformer layer
EMBED_DIM    = 1280        # ESM-2 650M output dimension
MAX_SEQ_LEN  = 1022        # ESM-2 hard limit (tokens)

print(f'Loading {MODEL_NAME} …')
print('(First run downloads ~1.3 GB from Meta servers)')

try:
    esm_model, alphabet = fair_esm.pretrained.esm2_t33_650M_UR50D()
    esm_model = esm_model.to(device).eval()
    print(f'\n✓ Model   : {MODEL_NAME}')
    print(f'✓ Device  : {device}')
    print(f'✓ Embed   : {EMBED_DIM} dimensions')
    print(f'✓ Max len : {MAX_SEQ_LEN} residues (longer sequences are truncated)')
except Exception as exc:
    log.error(f'Failed to load ESM-2 model: {exc}')
    raise


## Section 4 — Generate Embeddings (Optimised)

Embeddings are generated in sorted-length mini-batches for efficient padding.  
If a CUDA out-of-memory (OOM) error occurs the batch size is halved automatically.  
A checkpoint is written every `CHECKPOINT_INTERVAL` sequences so you can resume after interruption.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────
CHECKPOINT_FILE   = 'data/embeddings/embeddings_checkpoint.pt'
CHECKPOINT_META   = 'data/embeddings/metadata_checkpoint.csv'
CHECKPOINT_INTERVAL = 200          # save every N sequences

# Sensible defaults per device
BATCH_SIZE = 8 if device.type == 'cuda' else 2
MIN_BATCH  = 1

print(f'Batch size          : {BATCH_SIZE}')
print(f'Checkpoint interval : {CHECKPOINT_INTERVAL} sequences')
print(f'Mixed precision     : {device.type == "cuda"}')


In [ ]:
def embed_batch(
    sequences: List[str],
    model: torch.nn.Module,
    alphabet,
    device: torch.device,
    max_len: int = MAX_SEQ_LEN,
    repr_layer: int = REPR_LAYER,
) -> List[torch.Tensor]:
    """
    Compute mean-pooled ESM-2 embeddings for a list of sequences.

    Parameters
    ----------
    sequences  : Raw amino-acid strings.
    model      : Loaded ESM-2 model (already on *device*).
    alphabet   : ESM-2 alphabet object.
    device     : Target device.
    max_len    : Maximum number of residues (longer sequences are truncated).
    repr_layer : Transformer layer whose representations are used.

    Returns
    -------
    List of 1-D tensors, one per sequence, each of length *embed_dim*.
    """
    batch_converter = alphabet.get_batch_converter()
    truncated = [seq[:max_len] for seq in sequences]
    batch_data = [(f'seq_{i}', s) for i, s in enumerate(truncated)]

    _, _, tokens = batch_converter(batch_data)
    tokens = tokens.to(device)

    with torch.inference_mode():
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            out = model(tokens, repr_layers=[repr_layer], return_contacts=False)

    reps = out['representations'][repr_layer]  # (B, L+2, D)

    embeddings: List[torch.Tensor] = []
    for i, seq in enumerate(truncated):
        # Slice out only the sequence tokens (skip BOS at 0 and EOS at end)
        emb = reps[i, 1 : len(seq) + 1].mean(dim=0).cpu()
        embeddings.append(emb)

    return embeddings

print('✓ embed_batch() defined')


In [ ]:
def generate_all_embeddings(
    sequences: List[Dict],
    model: torch.nn.Module,
    alphabet,
    device: torch.device,
    batch_size: int = BATCH_SIZE,
    checkpoint_interval: int = CHECKPOINT_INTERVAL,
) -> Tuple[torch.Tensor, List[Dict]]:
    """
    Generate embeddings for every sequence with checkpointing and OOM recovery.

    Sequences are processed in ascending-length order for efficient GPU padding.
    A checkpoint is saved every *checkpoint_interval* processed sequences, so
    the pipeline can resume after an interruption without reprocessing.

    Parameters
    ----------
    sequences           : List of sequence dicts (keys: id, sequence, length, virus).
    model               : Loaded ESM-2 model.
    alphabet            : ESM-2 alphabet.
    device              : Target device.
    batch_size          : Initial mini-batch size (halved automatically on OOM).
    checkpoint_interval : Save progress every this many sequences.

    Returns
    -------
    (embeddings_tensor, metadata_list)
    """
    n_total = len(sequences)

    # ── Resume from checkpoint if it exists ──────────────────────────
    if Path(CHECKPOINT_FILE).exists() and Path(CHECKPOINT_META).exists():
        log.info('Checkpoint found — resuming…')
        saved_embs  = torch.load(CHECKPOINT_FILE, map_location='cpu')
        saved_meta  = pd.read_csv(CHECKPOINT_META).to_dict('records')
        done_ids    = {m['id'] for m in saved_meta}
        all_embs    = list(saved_embs)
        all_meta    = saved_meta
        print(f'✓ Resumed from checkpoint: {len(done_ids):,} sequences already done')
    else:
        done_ids: set = set()
        all_embs: List[torch.Tensor] = []
        all_meta: List[Dict] = []

    # ── Sort remaining sequences by length (efficient batching) ───────
    pending = sorted(
        [s for s in sequences if s['id'] not in done_ids],
        key=lambda x: x['length'],
    )
    n_pending = len(pending)

    if n_pending == 0:
        print('All sequences already embedded — loading from checkpoint.')
        return torch.stack(all_embs), all_meta

    print(f'\nGenerating embeddings for {n_pending:,} sequences…')
    print(f'Initial batch size: {batch_size}\n')

    pbar = tqdm(total=n_pending, unit='seq', desc='Embedding')
    bs   = batch_size
    idx  = 0

    try:
        while idx < n_pending:
            batch_seqs = pending[idx : idx + bs]
            seq_strs   = [s['sequence'] for s in batch_seqs]

            try:
                batch_embs = embed_batch(seq_strs, model, alphabet, device)
            except RuntimeError as exc:
                if 'out of memory' in str(exc).lower() and bs > MIN_BATCH:
                    bs = max(MIN_BATCH, bs // 2)
                    log.warning(f'OOM — reducing batch size to {bs}')
                    if device.type == 'cuda':
                        torch.cuda.empty_cache()
                    continue      # retry same batch with smaller bs
                raise            # re-raise non-OOM errors

            all_embs.extend(batch_embs)
            all_meta.extend(batch_seqs)

            idx  += len(batch_seqs)
            pbar.update(len(batch_seqs))

            # ── Clear GPU cache periodically ──────────────────────────
            if device.type == 'cuda':
                torch.cuda.empty_cache()

            # ── Checkpoint ────────────────────────────────────────────
            n_done = len(done_ids) + idx
            if (n_done % checkpoint_interval == 0) or (idx == n_pending):
                emb_tensor = torch.stack(all_embs)
                torch.save(emb_tensor, CHECKPOINT_FILE)
                pd.DataFrame(all_meta).to_csv(CHECKPOINT_META, index=False)
                pbar.write(
                    f'  ✓ Checkpoint saved — '
                    f'{n_done:,}/{n_total:,} seqs, '
                    f'tensor {tuple(emb_tensor.shape)}, '
                    f'{emb_tensor.nbytes/1e6:.1f} MB'
                )

    except KeyboardInterrupt:
        pbar.write('\n⚠ Interrupted — saving checkpoint before exit…')
        if all_embs:
            torch.save(torch.stack(all_embs), CHECKPOINT_FILE)
            pd.DataFrame(all_meta).to_csv(CHECKPOINT_META, index=False)
        pbar.close()
        raise

    pbar.close()

    embeddings = torch.stack(all_embs)
    return embeddings, all_meta

print('✓ generate_all_embeddings() defined')


In [ ]:
embeddings, final_metadata = generate_all_embeddings(
    all_seqs, esm_model, alphabet, device,
    batch_size=BATCH_SIZE,
    checkpoint_interval=CHECKPOINT_INTERVAL,
)

print('\n' + '=' * 70)
print('EMBEDDING GENERATION COMPLETE')
print('=' * 70)
print(f'✓ Shape  : {embeddings.shape}')
print(f'✓ Size   : {embeddings.nbytes / 1e9:.3f} GB')
print(f'✓ Range  : [{embeddings.min():.4f}, {embeddings.max():.4f}]')
print(f'✓ Mean   : {embeddings.mean():.6f}')
print(f'✓ Std    : {embeddings.std():.6f}')


## Section 5 — Save Embeddings

In [ ]:
# ── Save tensor ──────────────────────────────────────────────────────
torch.save(embeddings, 'data/embeddings/all_embeddings.pt')
print('✓ Saved: data/embeddings/all_embeddings.pt')

# ── Save NumPy array ─────────────────────────────────────────────────
np.save('data/embeddings/all_embeddings.npy', embeddings.numpy())
print('✓ Saved: data/embeddings/all_embeddings.npy')

# ── Save metadata ─────────────────────────────────────────────────────
meta_df = pd.DataFrame(final_metadata)
meta_df['embedding_idx'] = range(len(meta_df))
meta_df.to_csv('data/embeddings/embedding_metadata.csv', index=False)
print('✓ Saved: data/embeddings/embedding_metadata.csv')

# ── Save statistics ───────────────────────────────────────────────────
stats = {
    'model':               MODEL_NAME,
    'device':              str(device),
    'embedding_dimension': EMBED_DIM,
    'total_sequences':     int(embeddings.shape[0]),
    'lassa_count':         len(lassa_seqs),
    'ebola_count':         len(ebola_seqs),
    'embedding_mean':      float(embeddings.mean()),
    'embedding_std':       float(embeddings.std()),
    'embedding_min':       float(embeddings.min()),
    'embedding_max':       float(embeddings.max()),
    'seed':                SEED,
}

with open('data/embeddings/stats.json', 'w') as fh:
    json.dump(stats, fh, indent=2)
print('✓ Saved: data/embeddings/stats.json')

print(f'\nEmbedding Statistics:')
for k, v in stats.items():
    print(f'  {k:<25}: {v}')


## Section 6 — Clustering Analysis

In [ ]:
# ── Identify per-virus embedding ranges ──────────────────────────────
# The metadata list preserves insertion order; Lassa first, Ebola second.
lassa_mask = np.array([m['virus'] == 'Lassa' for m in final_metadata])
ebola_mask = ~lassa_mask

lassa_emb = embeddings[lassa_mask]
ebola_emb = embeddings[ebola_mask]

# ── Centroids ────────────────────────────────────────────────────────
lassa_centroid = lassa_emb.mean(dim=0)
ebola_centroid = ebola_emb.mean(dim=0)

euclid_dist = euclidean(lassa_centroid.numpy(), ebola_centroid.numpy())
cos_dist    = cosine(lassa_centroid.numpy(), ebola_centroid.numpy())

print('Centroid Distances')
print('-' * 40)
print(f'  Euclidean : {euclid_dist:.4f}')
print(f'  Cosine    : {cos_dist:.6f}')

# ── Within-cluster distances (sampled for speed) ──────────────────────
def within_cluster_distances(
    emb: torch.Tensor,
    centroid: torch.Tensor,
    n_sample: int = 200,
) -> np.ndarray:
    """Return Euclidean distances from a sample of embeddings to their centroid."""
    idx = np.random.choice(len(emb), size=min(n_sample, len(emb)), replace=False)
    return np.array([
        euclidean(emb[i].numpy(), centroid.numpy()) for i in idx
    ])

lassa_dists = within_cluster_distances(lassa_emb, lassa_centroid)
ebola_dists = within_cluster_distances(ebola_emb, ebola_centroid)

print(f'\nLassa within-cluster distances (n={len(lassa_dists)})')
print(f'  Mean : {lassa_dists.mean():.4f}  Std : {lassa_dists.std():.4f}')

print(f'\nEbola within-cluster distances (n={len(ebola_dists)})')
print(f'  Mean : {ebola_dists.mean():.4f}  Std : {ebola_dists.std():.4f}')

sep_ratio = euclid_dist / (lassa_dists.mean() + ebola_dists.mean())
print(f'\nSeparation ratio : {sep_ratio:.4f}')
status = ('EXCELLENT' if sep_ratio > 2
          else 'GOOD' if sep_ratio > 1
          else 'OVERLAPPING')
print(f'Status           : {status}')

# Append clustering stats to saved JSON
stats.update({
    'centroid_euclidean':   round(euclid_dist, 4),
    'centroid_cosine':      round(float(cos_dist), 6),
    'separation_ratio':     round(sep_ratio, 4),
    'clustering_status':    status,
})
with open('data/embeddings/stats.json', 'w') as fh:
    json.dump(stats, fh, indent=2)
print('\n✓ stats.json updated with clustering metrics')


## Section 7 — PCA Visualisation

In [ ]:
print('Fitting PCA…')
pca    = PCA(n_components=2, random_state=SEED)
emb_2d = pca.fit_transform(embeddings.numpy())

pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100
print(f'✓ PC1 variance : {pc1_var:.1f}%')
print(f'✓ PC2 variance : {pc2_var:.1f}%')
print(f'✓ Total        : {pc1_var + pc2_var:.1f}%')

lassa_2d = emb_2d[lassa_mask]
ebola_2d = emb_2d[ebola_mask]

fig, ax = plt.subplots(figsize=(12, 9))

ax.scatter(
    lassa_2d[:, 0], lassa_2d[:, 1],
    c='#FF6B6B', label=f'Lassa (n={lassa_mask.sum():,})',
    alpha=0.55, s=30, edgecolors='darkred', linewidths=0.3,
)
ax.scatter(
    ebola_2d[:, 0], ebola_2d[:, 1],
    c='#4ECDC4', label=f'Ebola (n={ebola_mask.sum():,})',
    alpha=0.55, s=30, edgecolors='navy', linewidths=0.3,
)

# Centroids in 2-D
lc = lassa_2d.mean(axis=0)
ec = ebola_2d.mean(axis=0)
ax.scatter(*lc, c='darkred',  marker='*', s=800, zorder=5,
           edgecolor='black', linewidth=1.5, label='Lassa centroid')
ax.scatter(*ec, c='navy',     marker='*', s=800, zorder=5,
           edgecolor='black', linewidth=1.5, label='Ebola centroid')
ax.plot([lc[0], ec[0]], [lc[1], ec[1]], 'k--', linewidth=1.5, alpha=0.4)

ax.set_xlabel(f'PC1 ({pc1_var:.1f}%)', fontweight='bold', fontsize=12)
ax.set_ylabel(f'PC2 ({pc2_var:.1f}%)', fontweight='bold', fontsize=12)
ax.set_title(
    f'PCA of ESM-2 Embeddings — Lassa vs Ebola\n'
    f'Separation ratio: {sep_ratio:.2f}  ({status})',
    fontsize=13, fontweight='bold',
)
ax.legend(fontsize=10)
ax.grid(alpha=0.25)

plt.tight_layout()
fig_path = 'results/figures/02_pca_visualization.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'✓ Saved: {fig_path}')
plt.show()


In [ ]:
# ── Embedding value distribution ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Embedding Value Distribution', fontsize=13, fontweight='bold')

sample = embeddings.numpy().flatten()
# Cap the flattened sample at 500 000 values: with 2 390 sequences × 1 280
# dimensions the full array is ~3 M values, which is fine, but for very
# large datasets this histogram would be extremely slow to render.
# 500 000 points give statistically identical visualisation while keeping
# notebook execution snappy on both CPU and GPU machines.
if len(sample) > 500_000:
    sample = np.random.choice(sample, 500_000, replace=False)

axes[0].hist(sample, bins=80, color='steelblue', alpha=0.75, edgecolor='none')
axes[0].set_xlabel('Embedding value', fontweight='bold')
axes[0].set_ylabel('Count', fontweight='bold')
axes[0].set_title('Distribution of all embedding values')
axes[0].grid(alpha=0.3)

# Per-class mean embedding norm
lassa_norms = lassa_emb.norm(dim=1).numpy()
ebola_norms = ebola_emb.norm(dim=1).numpy()
axes[1].hist(lassa_norms, bins=50, alpha=0.65, color='#FF6B6B', label='Lassa')
axes[1].hist(ebola_norms, bins=50, alpha=0.65, color='#4ECDC4', label='Ebola')
axes[1].set_xlabel('L2 norm', fontweight='bold')
axes[1].set_ylabel('Count', fontweight='bold')
axes[1].set_title('Embedding vector norms by class')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig_path = 'results/figures/03_embedding_distribution.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'✓ Saved: {fig_path}')
plt.show()


## Section 8 — Final Summary

In [ ]:
print('=' * 70)
print('✅  PHASE 3 COMPLETE — ESM-2 EMBEDDINGS')
print('=' * 70)

print(f'\n📊 Dataset')
print(f'   Lassa  : {len(lassa_seqs):,} sequences')
print(f'   Ebola  : {len(ebola_seqs):,} sequences')
print(f'   Total  : {len(all_seqs):,} sequences')

print(f'\n🧬 Embeddings')
print(f'   Model      : {MODEL_NAME}')
print(f'   Dimension  : {EMBED_DIM}')
print(f'   Device     : {device}')
print(f'   Shape      : {tuple(embeddings.shape)}')
print(f'   Size       : {embeddings.nbytes / 1e9:.3f} GB')

print(f'\n📈 Clustering Quality')
print(f'   Centroid Euclidean dist : {euclid_dist:.4f}')
print(f'   Centroid Cosine dist    : {cos_dist:.6f}')
print(f'   Separation ratio        : {sep_ratio:.4f}')
print(f'   Status                  : {status}')

print(f'\n💾 Saved Files')
for f in [
    'data/embeddings/all_embeddings.pt',
    'data/embeddings/all_embeddings.npy',
    'data/embeddings/embedding_metadata.csv',
    'data/embeddings/stats.json',
    'results/figures/01_sequence_length.png',
    'results/figures/02_pca_visualization.png',
    'results/figures/03_embedding_distribution.png',
]:
    exists = '✓' if Path(f).exists() else '✗'
    print(f'   {exists} {f}')

print(f'\n🚀 READY FOR ML MODELS (Phase 4)')
print('=' * 70)
